In [ ]:
# =======================================================
# Celda 1: Preparación del Entorno y Módulos
# =======================================================
import pandas as pd
import yaml
import os
import sys

# Configuración de rutas
sys.path.append(os.path.abspath('../src'))
from B01_negative_margin import MarginCorrector

with open('../config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

corrector = MarginCorrector(config)
print("✅ Entorno listo. Módulo de corrección de márgenes cargado.")

In [ ]:
# =======================================================
# Celda 2: Carga de Datos y Conversión de Fechas
# =======================================================
df_ventas = pd.read_csv("../" + config['paths']['raw_sales'])
df_precios = pd.read_csv("../" + config['paths']['raw_prices'])

# Guardamos el tipo original para el reporte
tipo_original = df_ventas['fecha'].dtype

# Conversión
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_precios['fecha'] = pd.to_datetime(df_precios['fecha'])

# NOTIFICAR AL JSON
corrector.registrar_cambio_formato("df_ventas", "fecha", tipo_original, df_ventas['fecha'].dtype)
corrector.registrar_cambio_formato("df_precios", "fecha", tipo_original, df_precios['fecha'].dtype)

print("✅ Cambio de formato registrado en la bitácora.")

In [ ]:
# =======================================================
# Celda 3: Corrección Inteligente y Generación de Evidencia
# =======================================================
# 1. Capturar índices con error
indices_error = df_precios[df_precios['precio_venta_unitario'] < df_precios['costo_unitario']].index

print(f"🔎 Analizando {len(indices_error)} registros sospechosos...")

# 2. Ejecutar corrección basada en Mediana Anual
df_precios_saneado = corrector.corregir_margenes_inteligente(df_precios)

# 3. Guardar el archivo JSON de evidencia
ruta_evidencia = corrector.generar_reporte_evidencia()

# 4. Mostrar comparativa visual
print("\n✅ PROCESO COMPLETADO")
print(f"📄 Evidencia guardada en: {ruta_evidencia}")
print("\n📊 COMPARATIVA DE REGISTROS AJUSTADOS (EJEMPLO):")

cols_view = ['producto_id', 'fecha', 'año', 'precio_venta_unitario', 'costo_unitario', 'margen_unitario']
print("\n--- ANTES DEL AJUSTE ---")
display(df_precios.loc[indices_error, cols_view])

print("\n--- DESPUÉS DEL AJUSTE (Mediana Anual) ---")
display(df_precios_saneado.loc[indices_error, cols_view])

In [ ]:
# =======================================================
# Celda 4: Exportación de Datos a Capa Interim
# =======================================================
# Crear carpeta interim si no existe
path_interim = "../data/interim/"
os.makedirs(path_interim, exist_ok=True)

# Guardar archivos listos para transformación
df_ventas.to_csv(os.path.join(path_interim, "df_ventas_imputed.csv"), index=False)
df_precios_saneado.to_csv(os.path.join(path_interim, "df_precios_imputed.csv"), index=False)

print(f"✅ Proceso completado. Archivos guardados en: {path_interim}")